In [1]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    ChatMessage,
    Role,
    WorkflowBuilder,
    WorkflowContext,
    ai_function,
    executor,
)

# 🤖 GitHub Models or OpenAI client integration
from agent_framework.openai import OpenAIChatClient
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")

✅ All imports successful!


## ಹಂತ 1: ರಚನೆಯಾದ ಔಟ್‌ಪುಟ್ಗಳಿಗಾಗಿ ಪೈಡ್ಯಾಂಟಿಕ್ ಮಾದರಿಗಳನ್ನು ವ್ಯಾಖ್ಯಾನಿಸಿ

ಈ ಮಾದರಿಗಳು ಏಜೆಂಟ್ಗಳು ಹಿಂತೆಗೆದುಕೊಳ್ಳುವ **ವಿನ್ಯಾಸ** ನಿಗದಿ ಮಾಡುತ್ತವೆ. ಪೈಡ್ಯಾಂಟಿಕ್ ಜೊತೆಗೆ `response_format` ಬಳಕೆ ಈನ್ನು ಖಚಿತಪಡಿಸುತ್ತದೆ:
- ✅ ಟೈಪ್-ಸೆಫ್ಫ್ ಡೇಟಾ ಹೊರತೆಗೆಯುವಿಕೆ
- ✅ ಸ್ವಯಂ ಪರಿಶೀಲನೆ
- ✅ ಉಚಿತ-ಪಠ್ಯ ಉತ್ತರಗಳಿಂದ ಪಾರ್ಸ್ ತಪ್ಪುಗಳು ಇಲ್ಲ
- ✅ ಕ್ಷೇತ್ರಗಳ ಆಧಾರದ ಮೇಲೆ ಸುಲಭ ತಿದ್ದುಪಡಿ ರೌಟಿಂಗ್


In [2]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

✅ Pydantic models defined:
   - BookingCheckResult (availability check)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)


## Step 2: ಹೋಟೆಲ್ ಬುಕ್ಕಿಂಗ್ ಟೂಲ್ ರಚಿಸಿ

ಈ ಟೂಲ್ ಅನ್ನು **availability_agent** ಕಾಲ್ ಮಾಡಿ ಕೊಠಡಿಗಳು ಲಭ್ಯವಿದೆಯೋ ಇಲ್ಲವೋ ಎಂದು ಪರಿಶೀಲಿಸುತ್ತದೆ. ನಾವು `@ai_function` ಡೆಕೊರೇಟರ್ ಅನ್ನು ಬಳಸುತ್ತೇವೆ:
- ಪೈಥಾನ್ ಫಂಕ್ಷನ್ ಅನ್ನು AI-ಕಾಲ್ ಮಾಡಬಹುದಾದ ಟೂಲ್ ಗೆ ಪರಿವರ್ತಿಸಲು
- LLM ಗಾಗಿ ಸ್ವಯಂಚಾಲಿತವಾಗಿ JSON ಸ್ಕೀಮಾ ರಚಿಸಲು
- ಪ್ಯಾರಾಮೀಟರ್ ಮಾನ್ಯತೆ ನಿಭಾಯಿಸಲು
- ಏಜೆಂಟ್ಗಳಿಂದ ಸ್ವಯಂಚಾಲಿತವಾಗಿ ഓഫಿಸುವಿಕೆಯನ್ನು ಸಕ್ರಿಯಗೊಳಿಸಲು

ಈ ಡೆಮೊಗಾಗಿ:
- **ಸ್ಟೋಕ್‌ಹೋಮ್, ಸिएಟಲ್, ಟೋಕಿಯೋ, ಲಂಡನ್, ಅಮ್ಸ್ಟರ್ಡ್ಯಾಮ್** → ಕೊಠಡಿಗಳು ಲಭ್ಯ ✅
- **ಇತರೆ ಎಲ್ಲ ನಗರಗಳು** → ಕೊಠಡಿಗಳು ಲಭ್ಯವಿಲ್ಲ ❌


In [3]:
@ai_function(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @ai_function decorator")

✅ hotel_booking tool created with @ai_function decorator


## ಹಂತ 3: ಮಾರ್ಗನಿರ್ದೇಷಣೆಗೆ ಶರತ್ತು ಕಾರ್ಯಗಳನ್ನು ವ್ಯಾಖ್ಯಾನಿಸಿ

ಈ ಕಾರ್ಯಗಳು ಏಜೆಂಟ್‌ನ ಪ್ರತಿಕ್ರಿಯೆಯನ್ನು ಪರಿಶೀಲಿಸಿ ವರ್ಕ್‌ಫ್ಲೋದಲ್ಲಿ ಯಾವ ಮಾರ್ಗವನ್ನು ತೆಗೆದುಕೊಳ್ಳಬೇಕೆಂದು ನಿರ್ಧರಿಸುತ್ತವೆ.

**ಮುಖ್ಯ ಮಾದರಿ:**
1. ಸಂದೇಶವು `AgentExecutorResponse` ಆಗಿದೆಯೇ ಅಂತ ಪರಿಶೀಲಿಸಿ
2. ರಚಿತ ಔಟ್‌ಪುಟ್ (Pydantic ಮಾದರಿ) ಅನ್ನು ಪಾರ್ಸ್ ಮಾಡಿ
3. ಮಾರ್ಗನಿರ್ದೇಷಣೆಯನ್ನು ನಿಯಂತ್ರಿಸಲು `True` ಅಥವಾ `False` ಅನ್ನು ಹಿಂತಿರುಗಿಸಿ

ವರ್ಕ್‌ಫ್ಲೋ ಈ ಶರತ್ತುಗಳನ್ನು **ಧಾರೆಗಳ** ಮೇಲೆ ಅಂಕಿಅಂಶಮಾಡಿ ಮುಂದಿನ ಯಾವ ಕಾರ್ಯಾಚರಣೆಯನ್ನು ಚಾಲನೆ ಮಾಡಬೇಕೆಂದು ನಿರ್ಧರಿಸುತ್ತದೆ.


In [4]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

✅ Condition functions defined:
   - has_availability_condition (routes when rooms exist)
   - no_availability_condition (routes when no rooms)


## ಹಂತ 4: ಕಸ್ಟಮ್ ಡಿಸ್ಪ್ಲೇ ಎಕ್ಸಿಕ್ಯೂಟರ್ ರಚಿಸಿ

ಎಕ್ಸಿಕ್ಯೂಟರ್‌ಗಳು ವರ್ಕ್‌ಫ್ಲೋ ಅಂಶಗಳಾಗಿದ್ದು, ಪರಿವರ್ತನೆಗಳು ಅಥವಾ ಪಾರ್ಶ್ವ ಪರಿಣಾಮಗಳನ್ನು ನೆರವೇರಿಸುತ್ತವೆ. ನಾವು ಅಂತಿಮ ಫಲಿತಾಂಶವನ್ನು ಪ್ರದರ್ಶಿಸುವ ಕಸ್ಟಮ್ ಎಕ್ಸಿಕ್ಯೂಟರ್ ರಚಿಸಲು `@executor` ಡೆಕೋರೇಟರ್ ಅನ್ನು ಬಳಸುತ್ತೇವೆ.

**ಮುಖ್ಯ ಕಲ್ಪನೆಗಳು:**
- `@executor(id="...")` - ಕಾರ್ಯವನ್ನು ವರ್ಕ್‌ಫ್ಲೋ ಎಕ್ಸಿಕ್ಯೂಟರ್ ಆಗಿ ನೋಂದಾಯಿಸುತ್ತದೆ
- `WorkflowContext[Never, str]` - ಇನ್‌ಪುಟ್/ಔಟ್‌ಪುಟ್‌ಗೆ ಪ್ರಕಾರ ಸೂಚನೆಗಳಾಗಿ
- `ctx.yield_output(...)` - ಅಂತಿಮ ವರ್ಕ್‌ಫ್ಲೋ ಫಲಿತಾಂಶ ನೀಡುತ್ತದೆ


In [5]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created with @executor decorator")

✅ display_result executor created with @executor decorator


## ಹಂತ 5: ಪರಿಸರ ಚರಗಳನ್ನು ಲೋಡ್ ಮಾಡಿ

LLM ಕ್ಲೈಎಂಟ್ ಅನ್ನು ಕಾನ್ಫಿಗರ್ ಮಾಡಿ. ಈ ಉದಾಹರಣೆ ಕಾರ್ಯನಿರ್ವಹಿಸುತ್ತದೆ:
- **GitHub ಮಾದರಿಗಳು** (GitHub ಟೋಕನ್‌ನೊಂದಿಗೆ ಉಚಿತ ತ.unregister ರಸ್ತೆ)
- **ಏಜ್ಯೂರ್ ಓಪನ್‌ಎಐ**
- **ಓಪನ್‌ಎಐ**


In [6]:
# Load environment variables
load_dotenv()

# Check for GitHub Models or OpenAI
chat_client = OpenAIChatClient(base_url=os.environ.get(
    "GITHUB_ENDPOINT"), api_key=os.environ.get("GITHUB_TOKEN"), model_id="gpt-4o")

## ಹಂತ 6: ಸಂರಚಿತ ಆದೇಶಗಳೊಂದಿಗೆ AI ಏಜೆಂಟುಗಳನ್ನು ರಚಿಸಿ

ನಾವು **ಮೂರು ವಿಶೇಷ ಏಜೆಂಟುಗಳನ್ನು** ರಚಿಸುತ್ತೇವೆ, ಪ್ರತಿ ಒಂದು `AgentExecutor` ನಲ್ಲಿ ಮುಡಿ ಬಿಗಿದುಕೊಳ್ಳುತ್ತದೆ:

1. **availability_agent** - ಸಾಧನವನ್ನು ಬಳಸಿ ಆತಿಥ್ಯ ಲಭ್ಯತೆ ಪರಿಶೀಲಿಸುತ್ತದೆ
2. **alternative_agent** - ಪರ್ಯಾಯ ನಗರಗಳನ್ನು ಸೂಚಿಸುತ್ತದೆ (ಬಡಿಗೆಗಳು ಇಲ್ಲದಾಗ)
3. **booking_agent** - ಬಡಿಗೆ ಲಭ್ಯವಿರುವಾಗ ಬುಕ್ಕಿಂಗ್ ಮಾಡಲು ಪ್ರೇರೇಪಿಸುತ್ತದೆ

**ಪ್ರಮುಖ ಲಕ್ಷಣಗಳು:**
- `tools=[hotel_booking]` - ಏಜೆಂಟ್‌ಗೆ ಸಾಧನವನ್ನು ನೀಡುತ್ತದೆ
- `response_format=PydanticModel` - ಸಂರಚಿತ JSON ಔಟ್‌ಪುಟ್ ಅನ್ನು ನಿಬಂಧಿಸುತ್ತದೆ
- `AgentExecutor(..., id="...")` - ಕಾರ್ಯಪ್ರವಾಹ ಬಳಕೆಗೆ ಏಜೆಂಟ್ ಅನ್ನು ಮುಡಿ ಹೊಡೆಯುತ್ತದೆ


In [7]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    chat_client.create_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        response_format=BookingCheckResult,
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    chat_client.create_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        response_format=AlternativeResult,
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    chat_client.create_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        response_format=BookingConfirmation,
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## ಹಂತ 7: ಷರತ್ತುಗೊಳಿಸಿದ ಧಾರಗಳಿಂದ ವರ್ಕ್ಫ್ಲೋ ಅನ್ನು ರಚಿಸು

ಈಗ ನಾವು `WorkflowBuilder` ಅನ್ನು ಬಳಸಿಕೊಂಡು ಷರತ್ತುಗೊಳಿಸಿದ ಮಾರ್ಗದರ್ಶನದೊಂದಿಗೆ ಗ್ರಾಫ್ ಅನ್ನು ನಿರ್ಮಿಸುತ್ತೇವೆ:

**ವರ್ಕ್ಫ್ಲೋ ರಚನೆ:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**ಪ್ರಮುಖ ವಿಧಾನಗಳು:**
- `.set_start_executor(...)` - ಪ್ರಾರಂಭ ಬಿಂದು ಹೊಂದಿಸುತ್ತದೆ
- `.add_edge(from, to, condition=...)` - ಷರತ್ತುಗೊಳಿಸಿದ ಧಾರೆಯನ್ನು ಸೇರಿಸುತ್ತದೆ
- `.build()` - ವರ್ಕ್ಫ್ಲೋ ಅನ್ನು ಅಂತಿಮಗೊಳಿಸುತ್ತದೆ


In [8]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder()
    .set_start_executor(availability_agent)
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## Step 8: ಟೆಸ್ಟ್‌ ಕೇಸ್ 1 ಚಲಾಯಿಸಿ - ನಗರದಲ್ಲಿ ಲಭ್ಯತೆ ಇಲ್ಲ (ಪ್ಯಾರಿಸ್)

ನಾವು ನಮ್ಮ ಅನುಕರಣದಲ್ಲಿ ಯಾವುದೇ ಕೋಣೆಗಳು ಇಲ್ಲದ ಪ್ಯಾರಿಸ್‌ನಲ್ಲಿ ಹೋಟೆಲ್ಗಳನ್ನು ಕೇಳಿಕೊಂಡು **ಲಭ್ಯತೆ ಇಲ್ಲದ** ಮಾರ್ಗವನ್ನು ಪರೀಕ್ಷಿಸೋಣ.


In [9]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[ChatMessage(Role.USER, text="I want to book a hotel in Paris")], should_respond=True
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## ಹಂತ 9: ಪರೀಕ್ಷಾ ಪ್ರಕರಣ 2 ರನ್ ಮಾಡಿ - ನಗರ WITH ಲಭ್ಯತೆ (ಸ್ಟಾಕ್‌ಹೋಮ್)

ಈಗ ನಾವು **ಲಭ್ಯತೆ** ಮಾರ್ಗವನ್ನು ಸ್ಟಾಕ್‌ಹೋಮ್‌ನಲ್ಲಿ ಹೋಟೆಲ್‌ಗಳನ್ನು ವಿನಂತಿಸುವ ಮೂಲಕ ಪರೀಕ್ಷಿಸೋಣ (ನಮ್ಮ ಅನುಕರಣದಲ್ಲಿ ಕೊಠಡಿಗಳು ಲಭ್ಯವಿವೆ).


In [10]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[ChatMessage(Role.USER, text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## ಮುಖ್ಯ ಅಂಶಗಳು ಮತ್ತು ಮುಂದಿನ ಹಂತಗಳು

### ✅ ನೀವು ಕಲಿತಿರುವುದು:

1. **WorkflowBuilder ಪ್ಯಾಟರ್ನ್**
   - ಪ್ರವೇಶ ಬಿಂದುವನ್ನು ನಿರ್ಧರಿಸಲು `.set_start_executor()` ಬಳಸಿ
   - ಷರತ್ತು ಆಧಾರಿತ ರುಟಿಂಗ್‌ಗಾಗಿ `.add_edge(from, to, condition=...)` ಬಳಕಿ
   - ವರ್ಕ್‌ಫ್ಲೋ ಅನ್ನು ಅಂತಿಮಗೊಳಿಸಲು `.build()` ಕರೆಮಾಡಿ

2. **ಷರತ್ತು ಆಧಾರಿತ رುಟಿಂಗ್**
   - ಶರತ್ತು ಫಂಕ್ಷನ್‌ಗಳು `AgentExecutorResponse` ಅನ್ನು ಪರಿಶೀಲಿಸುತ್ತವೆ
   - ರುಟಿಂಗ್ ನಿರ್ಧಾರ ಮಾಡಲು ಸಂರಚಿತ ಔಟ್‌ಪುಟ್‌ಗಳನ್ನು ವಿಶ್ಲೇಷಿಸಿ
   - ಒಂದು ಎಡ್ಜ್‌ನ್ನು ಸಕ್ರಿಯಗೊಳಿಸಲು `True`, ಅದನ್ನು ತಪ್ಪಿಸಲು `False` ವಾಪಸ್ ಮಾಡುತ್ತದೆ

3. **ಟೂಲ್ ಏಕೀಕರಣ**
   - ಪೈಥಾನ್ ಫಂಕ್ಷನ್‌ಗಳನ್ನು AI ಟೂಲ್‌ಗಳಾಗಿ ಪರಿವರ್ತಿಸಲು `@ai_function` ಬಳಸಿ
   - ಅಗತ್ಯವಿದ್ದಾಗ ಏಜೆಂಟ್‌ಗಳು ಟೂಲ್‌ಗಳನ್ನು ಸ್ವಯಂಚಾಲಿತವಾಗಿ ಕರೆಮಾಡುತ್ತವೆ
   - ಟೂಲ್‌ಗಳು ಏಜೆಂಟ್‌ಗಳು ವಿಶ್ಲೇಷಿಸಬಹುದು ಅಂತ JSON ವಾಪಸ್ ಮಾಡುತ್ತವೆ

4. **ಸಂರಚಿತ ಔಟ್‌ಪುಟ್‌ಗಳು**
   - ತಪಾಸಿತ ಸಂಗ್ರಹಣೆಯ 데이터 ಪಡೆಯಲು ಪೈಡಾಂಟಿಕ್ ಮಾದರಿಗಳನ್ನು ಬಳಸಿ
   - ಏಜೆಂಟ್‌ಗಳನ್ನು ರಚಿಸುವಾಗ `response_format=MyModel` ಸೆಟ್ ಮಾಡಿ
   - ಪ್ರತಿಕ್ರಿಯೆಗಳನ್ನು `Model.model_validate_json()` ಮೂಲಕ ವಿಶ್ಲೇಷಿಸಿ

5. **ಕಸ್ಟಮ್ ಎಕ್ಸಿಕ್ಯೂಟರ್‌ಗಳು**
   - ವರ್ಕ್‌ಫ್ಲೋ ಘಟಕಗಳನ್ನು ರಚಿಸಲು `@executor(id="...")` ಬಳಸಿ
   - ಎಕ್ಸಿಕ್ಯೂಟರ್‌ಗಳು ಡೇಟಾ ಪರಿವರ್ತನೆ ಅಥವಾ ಪಾರ್ಶ್ವ ಪರಿಣಾಮಗಳನ್ನು ರೂಪಿಸಬಹುದು
   - ವರ್ಕ್‌ಫ್ಲೋ ಫಲಿತಾಂಶಗಳ ಉತ್ಪಾದನೆಗೆ `ctx.yield_output()` ಬಳಸಿ

### 🚀 ವಾಸ್ತವ ಜಗತ್ತಿನ ಅನ್ವಯಿಕೆಗಳು:

- **ಪ್ರಯಾಣ ಬುಕ್ಕಿಂಗ್**: ಲಭ್ಯತೆ ಪರಿಶೀಲನೆ, ಪರ್ಯಾಯವನ್ನು ಸೂಚಿಸುವುದು, ಆಯ್ಕೆಗಳನ್ನು ಹೋಲಿಕೆ ಮಾಡುವುದು
- **ಗ್ರಾಹಕ ಸೇವೆ**: ಸಮಸ್ಯೆಯ ಪ್ರಕಾರ, ಭಾವನೆ, ಆದ್ಯತೆ ಆಧಾರಿತ ರುಟಿಂಗ್
- **ಇ-ಕಾಮರ್ಸ್**: ಜಾಗತಿಕತೆಯ ಪರಿಶೀಲನೆ, ಪರ್ಯಾಯಗಳನ್ನು ಸೂಚಿಸುವುದು, ಆರ್ಡರ್‌ಗಳನ್ನು ಪ್ರಕ್ರಿಯೆಗೊಳಿಸುವುದು
- **ವಿಷಯೋದ್ಯಮ ನಿಯಂತ್ರಣ**: ವಿಷಕಾರಿ ಅಂಶಗಳು, ಬಳಕೆದಾರ ಫ್ಲಾಗ್ ಆಧಾರಿತ ರುಟಿಂಗ್
- **ಅನುಮೋದನೆ ವರ್ಕ್‌ಫ್ಲೋಗಳು**: ಮೊತ್ತ, ಬಳಕೆದಾರರ ಪಾತ್ರ, ಅಪಾಯ ಮಟ್ಟ ಆಧಾರಿತ ರುಟಿಂಗ್
- **ಬಹು ಹಂತದ ಪ್ರಕ್ರಿಯೆ**: ಡೇಟಾ ಗುಣಮಟ್ಟ, ಪೂರ್ಣತೆ ಆಧಾರಿತ ರುಟಿಂಗ್

### 📚 ಮುಂದಿನ ಹಂತಗಳು:

- ಹೆಚ್ಚು ಸಂಕೀರ್ಣ ಷರತ್ತುಗಳನ್ನು ಸೇರಿಸುವುದು (ಬಹು ಮಾನದಂಡಗಳು)
- ವರ್ಕ್‌ಫ್ಲೋ ಸ್ಥಿತಿವ್ಯವಸ್ಥೆ ಸಹಿತ ಲೂಪ್‌ಗಳನ್ನು ಅನುಷ್ಠಾನಗೊಳಿಸುವುದು
- ಮರುಬಳಕೆ ಯೋಗ್ಯ ಘಟಕಗಳಿಗೆ ಉಪ-ವರ್ಕ್‌ಫ್ಲೋಗಳನ್ನು ಸೇರಿಸುವುದು
- ನೈಜ APIಗಳೊಂದಿಗೆ ಏಕೀಕರಣ (ಹೋಟೆಲ್ ಬುಕ್ಕಿಂಗ್, ಜಾಗತಿಕತೆಯ ವ್ಯವಸ್ಥೆಗಳು)
- ದೋಷ ನಿರ್ವಹಣೆ ಮತ್ತು ಬ್ಯಾಕ್ಅಪ್ ಮಾರ್ಗಗಳನ್ನು ಸೇರಿಸುವುದು
- ಒಳಗೊಂಡ ದೃಶ್ಯೀಕರಣ ಉಪಕರಣಗಳೊಂದಿಗೆ ವರ್ಕ್‌ಫ್ಲೋಗಳನ್ನು ದೃಶ್ಯೀಕರಿಸುವುದು


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ನಿರಾಕರಣೆ**:  
ಈ ಡಾಕ್ಯುಮೆಂಟ್ ಅನ್ನು AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಗೆ ಪ್ರಯತ್ನಿಸುತ್ತಿದ್ದರೂ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ತಪ್ಪುಗಳು ಅಥವಾ ಅಸಮರ್ಪಕತೆಗಳು ಇರಬಹುದೆಂದು ದಯವಿಟ್ಟು ಗಮನಿಸಿರಿ. ಮೂಲ ಭಾಷೆಯಲ್ಲಿ ಇರುವ ಡಾಕ್ಯುಮೆಂಟ್ ಅನ್ನು ಪ್ರಾಧಿಕಾರಿತ ಮೂಲವೆಂದು ಪರಿಗಣಿಸುವುದು ಮುಖ್ಯ. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದದ ಬಳಕೆಮೂಲಕ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪುಮಾತು ಅಥವಾ ಭ್ರಮೆಗಳಿಗಾಗಿಯೂ ನಾವು ಜವಾಬ್ದಾರಿಯಾಗುವುದಿಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
